# Lab03 — Thí nghiệm Hugging Face: 10 epoch, validation selection và artifacts

Notebook chạy từ đầu trên **Kaggle GPU**, gồm cả Exercise 1 và Exercise 2. Mỗi candidate DistilBERT chạy **đủ 10 epoch**, không early stopping; đánh giá mô hình tốt nhất, không mặc định lấy epoch cuối. Không có số liệu 10 epoch được điền sẵn trước khi chạy.

Bật **GPU T4** và **Internet**. Nếu chọn T4 x2, notebook chỉ dùng GPU đầu tiên để giữ thí nghiệm đơn giản, dễ tái lập. Chạy tuần tự các cell; sau khi đổi accelerator hoặc restart session, chạy lại setup. Không chạy notebook lịch sử đồng thời.

| Phase | Công việc / quyết định | Module tham chiếu | Artifact và hình hiển thị |
|---|---|---|---|
| P0 | Clone repo, Python 3.12 riêng, kiểm tra GPU | requirements.txt, config/config.json | environment snapshot, GPU info |
| P1 | Pytest, sanity overfit trên fixture nhỏ | tests/test_training.py, tests/test_study.py | pytest XML + log |
| P2 | Pipeline, tokenizer, manual forward | src/workflow.py, src/models/classifier.py | token heatmap, bảng dự đoán |
| P3 | Full official splits, audit overlap | src/datasets/loader.py, src/study.py | split_overlap_audit.json |
| P4 | TF-IDF + logistic regression | src/study.py:fit_text_baseline | validation candidates và policy |
| P5 | LR 1e-5 / 2e-5 / 3e-5, mỗi LR đủ 10 epoch | src/training/trainer.py, config/study.json | curves theo epoch và step cho từng LR |
| P6 | Chốt candidate bằng validation Macro-F1 | src/study.py:choose_candidate | frozen_selection.json |
| P7 | Threshold 0.05…0.95 trên validation | src/training/thresholds.py | threshold curves, decision_threshold.json |
| P8 | Đánh giá test sau khi khóa selection | src/study.py:run_study | metrics, confusion matrices, ROC/PR, calibration, CI |
| P9 | Render trực tiếp và xuất hai gói ZIP | scripts/display_study.py, scripts/package_study.py | review ZIP nhẹ; checkpoint ZIP riêng |

Tài liệu: [description](../description_lab_03.md), [walkthrough](../walkthrough_lab_03.md), [README](../README.md). Các playbook tham chiếu: [DL flow](../../../docs/dl_flow.md), [DL data](../../../docs/dl_data.md), [DL training](../../../docs/dl_training.md).

**Đã có phiên chạy thật:** xem [05_study_results_and_discussion.ipynb](05_study_results_and_discussion.ipynb) để đọc đầy đủ ảnh, số liệu và thảo luận của study 10 epoch. Chỉ chạy template này khi muốn tạo thí nghiệm GPU mới.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink

REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
BRANCH = "lab03"
REPO = Path("/kaggle/working/Lab-DL")
ENV = Path("/kaggle/working/lab03-py312")
PYTHON = str(ENV / "bin/python")

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Hãy bật Settings > Accelerator > GPU và chạy lại từ cell đầu.")
subprocess.run(["nvidia-smi"], check=True)

if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    print("Repo đã tồn tại; giữ nguyên file và kết quả hiện có.")
LAB = REPO / "labs/lab03"
assert (LAB / "scripts/run_study.py").is_file(), "Repo chưa có bản 10 epoch; cập nhật branch lab03 trước khi chạy."
print("Code revision:")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.5"], check=True)
def uv(*args):
    subprocess.run([sys.executable, "-m", "uv", *map(str, args)], check=True)
uv("python", "install", "3.12")
if not Path(PYTHON).is_file():
    uv("venv", "--python", "3.12", ENV)
uv("pip", "install", "--python", PYTHON, "--link-mode", "copy", "-r", LAB / "requirements.txt")
uv("pip", "check", "--python", PYTHON)

RUN_ENV = os.environ.copy()
RUN_ENV.update(CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
               CUBLAS_WORKSPACE_CONFIG=":4096:8", MPLCONFIGDIR="/kaggle/working/.mpl-lab03")
def run_python(*args, env=None):
    return subprocess.run([PYTHON, "-u", *map(str, args)], cwd=REPO,
                          env=RUN_ENV if env is None else env, check=True)
run_python("-c", "import sys, torch, transformers; print(sys.version); print('torch', torch.__version__); print('transformers', transformers.__version__); assert torch.cuda.is_available(), 'GPU chưa khả dụng'; print(torch.cuda.get_device_name(0))")
free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Dung lượng trống: {free_gib:.1f} GiB")
if free_gib < 6:
    raise RuntimeError("Cần ít nhất 6 GiB trống cho candidate checkpoints và ZIP. Hãy dùng session sạch.")

## P1 — Kiểm thử trước khi dùng GPU

Các tests dùng mô hình DistilBERT cực nhỏ và dữ liệu tổng hợp; đây là kiểm thử pipeline, không phải benchmark. Cell này ép CPU cho tests, có bài kiểm tra đủ 10 epoch trên fixture nhỏ và sanity overfit. Mọi test phải pass trước khi chạy study. Pytest XML và log được lưu vào `outputs/`.

In [ ]:
from datetime import datetime, timezone
validation_dir = LAB / "outputs/preflight" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_dir.mkdir(parents=True, exist_ok=False)
TEST_ENV = RUN_ENV.copy()
TEST_ENV["CUDA_VISIBLE_DEVICES"] = ""
with (validation_dir / "pytest.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen([PYTHON, "-u", "-m", "pytest", "labs/lab03/tests", "-q",
                                f"--junitxml={validation_dir / 'pytest.xml'}"],
                               cwd=REPO, env=TEST_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=""); log.write(line)
    if process.wait() != 0:
        raise RuntimeError("Preflight tests failed. Chưa chạy training; xem pytest.log.")
print("Preflight passed:", validation_dir)

## P2 — Exercise 1: pretrained sentiment analysis

Tokenize một câu, đối chiếu pipeline với manual forward/softmax, rồi hiển thị ảnh từ chính artifacts đã lưu. Không dùng mô hình SST-2 này làm backbone cho Exercise 2: Exercise 2 khởi tạo classification head mới từ DistilBERT base.

In [ ]:
run_python("labs/lab03/scripts/run_exercise1.py")
run_python("-c", "from labs.lab03.src.analysis import ensure_exercise1_analysis; from labs.lab03.src.utils.runs import ROOT; import json; s=json.loads((ROOT/'config/workflow.json').read_text()); print(ensure_exercise1_analysis(ROOT/'outputs/runs'/s['active_runs']['exercise1']))")
state = json.loads((LAB / "config/workflow.json").read_text())
exercise1_run = LAB / "outputs/runs" / state["active_runs"]["exercise1"]
display(json.loads((exercise1_run / "predictions.json").read_text()))
exercise1_analysis = LAB / "outputs/runs" / state["active_runs"]["exercise1_analysis"]
for image_path in sorted(exercise1_analysis.glob("*.png")):
    display(Markdown(f"**{image_path.name}**"))
    display(Image(filename=str(image_path)))

## P3–P8 — Train đủ 10 epoch và chốt mô hình bằng validation

- Dữ liệu nguyên vẹn: **8.530 train / 1.066 validation / 1.066 test**.
- TF-IDF fit từ vựng và IDF chỉ trên train; chọn C trong `[0.1, 1, 10]` bằng validation Macro-F1, hòa thì ưu tiên validation loss thấp hơn rồi C nhỏ hơn.
- DistilBERT: mỗi LR `[1e-5, 2e-5, 3e-5]` chạy **đủ 10 epoch**; tổng cộng 30 epoch giữa ba candidate. Không early stopping. Cùng seed 42 để so cấu hình có kiểm soát.
- Log mỗi 25 optimizer step, validation/checkpoint mỗi 100 step và cuối mọi epoch. Với batch 16 trên một GPU, có 534 step/epoch và 5.340 step/candidate. Full train-eval cuối epoch để so loss/accuracy ở cùng chế độ eval với validation. Nếu AMP bỏ qua update do overflow, selection_summary.json ghi riêng số skipped steps và successful optimizer updates.
- AdamW, warmup 10%, linear decay, weight decay 0.01, gradient clipping 1.0. FP16 cho T4, dynamic padding theo bội số 8. Không dùng torch.compile vì overhead biên dịch chưa được chứng minh có lợi trên workload này.
- Chọn checkpoint theo validation Macro-F1 (argmax); hòa trong cùng candidate giữ checkpoint đạt đầu tiên. Chọn LR theo Macro-F1, hòa thì validation loss thấp hơn rồi LR nhỏ hơn.
- Threshold sweep mặc định 19 điểm; chốt bằng validation Macro-F1, hòa ưu tiên gần 0.5 rồi threshold nhỏ hơn. Không chọn threshold trên test.
- Candidate không tạo dự đoán test. Chỉ winner và baseline đã chốt được đánh giá trên test sau cùng.

**Không hứa accuracy tăng chỉ vì chạy lâu hơn.** Kết quả cũ đã có dấu hiệu overfit; checkpoint selection giữ lại mô hình tốt nhất trong toàn bộ quá trình. Test lịch sử từng được xem nên đây là thí nghiệm mở rộng, không được gọi là benchmark test hoàn toàn chưa từng đụng tới. Một seed chưa đủ đánh giá biến thiên do khởi tạo.

In [ ]:
config = json.loads((LAB / "config/config.json").read_text())
study_settings = json.loads((LAB / "config/study.json").read_text())
assert config["exercise2"]["num_train_epochs"] == 10
assert study_settings["epochs_per_candidate"] == 10
print(json.dumps(study_settings, indent=2))

# Chạy cell này đúng một lần cho mỗi thí nghiệm mới.
# Chạy lại sẽ tạo study và candidate mới, không ghi đè hoặc resume run cũ.
run_python("labs/lab03/scripts/run_study.py")
state = json.loads((LAB / "config/workflow.json").read_text())
STUDY = LAB / "outputs/runs" / state["active_runs"]["study"]
print("Study đã hoàn tất:", STUDY)

## P9 — Visualize trực tiếp trong notebook

Cell sau chạy trong kernel notebook hiện tại, không trong subprocess. Các PNG hiển thị chính là file đã lưu vào `outputs/runs/`, gồm mọi candidate, learning curves theo epoch/step, validation sweep, baseline comparison, confusion matrices, ROC/PR, calibration và mẫu dự đoán.

Đọc train-eval loss và validation loss cùng chế độ; optimization loss là trung bình các mini-batch khi dropout bật và trọng số đang cập nhật nên được vẽ riêng. Validation loss tăng khi accuracy đứng yên thường là mô hình sai tự tin hơn. Không làm mượt để che hiện tượng này.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("lab03_display_study", LAB / "scripts/display_study.py")
viewer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(viewer)
shown_figures = viewer.display_study(STUDY, root=LAB)
assert len(shown_figures) >= 10, "Thiếu hình trực quan; kiểm tra artifacts trước khi đóng gói."

## Đọc kết quả và thảo luận

1. LR nào thắng trên validation? Checkpoint tốt nhất nằm ở epoch/step nào, có khác epoch 10 không?
2. Từ lúc nào train-eval loss tiếp tục giảm còn validation loss tăng? Có NaN/gradient bất thường không?
3. DistilBERT có hơn TF-IDF trên cùng test không? Mức tăng accuracy và Macro-F1 có cùng chiều không?
4. Threshold làm FP/FN đổi ra sao? Recall giảm có chấp nhận được với mục tiêu bài toán không?
5. ROC-AUC tốt có đồng nghĩa confidence calibrated tốt không? Xem Brier score, NLL và reliability.
6. CI bootstrap phản ánh bất định lấy mẫu test cho một mô hình cố định; không phải CI cho mọi lần train.
7. Có overlap exact/normalized giữa split không? Nếu có, báo cáo rõ, không tự xóa làm thay đổi benchmark.
8. Các câu sai là phủ định, tương phản, mỉa mai hay từ vựng nội dung phim? Giải thích ngôn ngữ là giả thuyết, không khẳng định nguyên nhân mô hình.

Không sửa learning rate, epoch hoặc threshold để tối đa test sau khi xem kết quả. Nếu tiếp tục nghiên cứu, dùng validation và báo cáo việc test đã từng được xem.

## Đóng gói để review và giữ checkpoint

- **Review ZIP**: số liệu, PNG, dự đoán, policy, cấu hình, log nguyên byte, manifest của study và các candidate. Gửi gói này để rà soát và hoàn thiện báo cáo. Không chứa trọng số.
- **Checkpoint ZIP**: chỉ best model của winner, tokenizer, cấu hình và policy. Tải về giữ để inference, reload và phân tích bổ sung mà không phải train lại. Không commit vào Git thông thường.
- Notebook đang mở do Kaggle quản lý: **Save Version**, rồi **File → Download Notebook** để giữ cả output tương tác. Review ZIP không tự lấy được nội dung notebook đang mở từ trình duyệt.

Mọi output gốc nằm dưới `labs/lab03/outputs/`; hai ZIP được sao chép lên `/kaggle/working/` để hiện rõ trong panel Output và tải xuống.

In [ ]:
run_python("labs/lab03/scripts/audit_study.py", "--study-run", STUDY)
run_python("labs/lab03/scripts/package_study.py", "--study-run", STUDY)
bundle = json.loads((LAB / "outputs/bundles/latest_bundle.json").read_text())
for key in ["review", "checkpoint"]:
    if not bundle.get(key):
        continue
    source = Path(bundle[key])
    download = Path("/kaggle/working") / source.name
    shutil.copy2(source, download)
    print(key, ":", download, f"({download.stat().st_size / 1024**2:.1f} MB)")
    display(FileLink(str(download.relative_to(Path.cwd())) if download.is_relative_to(Path.cwd()) else str(download)))
print("SHA256 review:", bundle["review_sha256"])
print("Nếu panel Output chưa hiện ZIP: nhấn refresh cạnh /kaggle/working.")